# Tempo de sentença nos Juizados Especiais

Neste notebook baixei diretamente a base Justiça em Números e separei o tempo médio da sentença na fase de conhecimento dos Juizados Especiais dos 27 Tribunais de Justiça. Como o CNJ disponibiliza o arquivo por um link direto, não foi necessário utilizar Selenium.

In [2]:
import zipfile
from pathlib import Path

import pandas as pd
import requests

#projeto feito com o auxilio de IA Chat GPT
PASTA_PROJETO = Path.cwd()
if PASTA_PROJETO.name == "src":
    PASTA_PROJETO = PASTA_PROJETO.parent
ARQUIVO_BRUTO = PASTA_PROJETO / "data" / "raw" / "justica_em_numeros_2026.zip"
ARQUIVO_TRATADO = PASTA_PROJETO / "data" / "processed" / "tempo_sentenca_juizados.csv"

In [3]:
# Download da base publicada pelo CNJ em 23 de junho de 2026.
resposta = requests.get(
    "https://www.cnj.jus.br/wp-content/uploads/2026/06/23-jun-2026.zip",
    headers={"User-Agent": "Mozilla/5.0 (projeto acadêmico TP3)"},
    timeout=180,
)
print("Status do download:", resposta.status_code)
resposta.raise_for_status()
ARQUIVO_BRUTO.write_bytes(resposta.content)

Status do download: 200


4248648

In [4]:
# Leitura do arquivo principal que estava dentro do ZIP.
with zipfile.ZipFile(ARQUIVO_BRUTO) as arquivo_zip:
    print("Arquivos encontrados:", arquivo_zip.namelist())
    dados = pd.read_csv(
        arquivo_zip.open("JN_23-Jun-2026.csv"),
        sep=";",
        encoding="cp1252",
        decimal=",",
        na_values=["nd", "n/a"],
        low_memory=False,
    )

print("Linhas e colunas da base:", dados.shape)
dados[["ano", "justica", "sigla", "tpsentcjem"]].tail()

Arquivos encontrados: ['Variaveis.csv', 'BD_Consolidado_JF_Secao_23_Jun_2026.csv', 'JN_23-Jun-2026.csv']
Linhas e colunas da base: (1595, 1314)


,ano,justica,sigla,tpsentcjem
1590,2025,Trabalho,TRT21,NaN
1591,2025,Trabalho,TRT22,NaN
1592,2025,Trabalho,TRT23,NaN
1593,2025,Trabalho,TRT24,NaN
1594,2025,Trabalho,TRT,NaN


In [5]:
# Recorte da Justiça Estadual no ano de 2025.
tempo_sentenca = dados.loc[
    (dados["ano"] == 2025)
    & (dados["justica"] == "Estadual")
    & (dados["sigla"] != "TJ"),
    ["ano", "sigla", "dsc_tribunal", "uf_sede", "tpsentcjem", "sentcje"],
].copy()

tempo_sentenca.columns = [
    "ano",
    "tribunal",
    "nome_tribunal",
    "uf",
    "tempo_medio_sentenca_dias",
    "sentencas_conhecimento",
]
tempo_sentenca["tempo_medio_sentenca_dias"] = tempo_sentenca["tempo_medio_sentenca_dias"].round(1)
tempo_sentenca = tempo_sentenca.sort_values("tempo_medio_sentenca_dias").reset_index(drop=True)
tempo_sentenca

,ano,tribunal,nome_tribunal,uf,tempo_medio_sentenca_dias,sentencas_conhecimento
0,2025,TJDFT,TJ - Distrito Federal,DF,90.1,138350.0
1,2025,TJSE,TJ - Sergipe,SE,111.7,47683.0
2,2025,TJBA,TJ - Bahia,BA,120.0,534267.0
3,2025,TJMA,TJ - Maranhão,MA,125.7,67125.0
4,2025,TJAC,TJ - Acre,AC,129.6,20584.0
5,2025,TJAM,TJ - Amazonas,AM,136.5,217044.0
6,2025,TJRO,TJ - Rondônia,RO,136.5,58533.0
7,2025,TJRR,TJ - Roraima,RR,139.7,18259.0
8,2025,TJGO,TJ - Goiás,GO,144.6,220578.0
9,2025,TJMT,TJ - Mato Grosso,MT,157.1,130102.0


In [6]:
# Testes feitos antes de salvar o resultado.
assert len(tempo_sentenca) == 27
assert tempo_sentenca["tribunal"].nunique() == 27
assert tempo_sentenca["tempo_medio_sentenca_dias"].notna().all()
assert (tempo_sentenca["tempo_medio_sentenca_dias"] > 0).all()

print("Testes concluídos sem erros.")
print("Menor tempo:", tempo_sentenca.iloc[0]["tribunal"], tempo_sentenca.iloc[0]["tempo_medio_sentenca_dias"])
print("Maior tempo:", tempo_sentenca.iloc[-1]["tribunal"], tempo_sentenca.iloc[-1]["tempo_medio_sentenca_dias"])

Testes concluídos sem erros.
Menor tempo: TJDFT 90.1
Maior tempo: TJRS 414.2


In [7]:
tempo_sentenca.to_csv(ARQUIVO_TRATADO, index=False, encoding="utf-8-sig")
print("Arquivo bruto:", ARQUIVO_BRUTO)
print("Arquivo tratado:", ARQUIVO_TRATADO)

Arquivo bruto: c:\Carol\Facul\2 ano - 2 bi\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP3\PB_Juizados_Especiais\data\raw\justica_em_numeros_2026.zip
Arquivo tratado: c:\Carol\Facul\2 ano - 2 bi\Projeto de Bloco\CarolinaGabrielaMesquita_PB_TP3\PB_Juizados_Especiais\data\processed\tempo_sentenca_juizados.csv
